# Transfer Learning + DAFE: Before vs After

## Goal
Compare: Transfer Learning only (79.4% mAP) vs Transfer Learning + DAFE module

## Pipeline
```
Step 1: Load transfer learning best weights (79.4% mAP) — reference
Step 2: Build DAFE architecture (digisteel.yaml) + load COCO pretrained weights
Step 3: Freeze early backbone (0-6), unfreeze DAFE + neck + head (7-24)
Step 4: Train with optimized recipe (same as exp_5a)
Step 5: Evaluate on TEST set
Step 6: Compare: TL only vs TL + DAFE
```

## Key Fix
Previous version failed because C3k2 (yolo11n) ≠ C2f (digisteel.yaml) — manual weight transfer didn't work.
This version uses `model.load('yolo11n.pt')` — Ultralytics automatically matches compatible layers.

## Expected
- TL only: 79.4% mAP@0.5 (reference)
- TL + DAFE: 80-83% mAP@0.5 (DAFE adds defect-aware enhancement)

In [ ]:
# ============================================================
# Cell 2: Setup
# ============================================================
import torch
import ultralytics
from pathlib import Path
import json
import sys
import random
import numpy as np

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Add project root to path for custom modules
ROOT = Path(r"D:/DigiSteel-Yolo/DigiSteel-YOLO")
sys.path.insert(0, str(ROOT))

# Paths
DATA_YAML = ROOT / "configs" / "data" / "neu_det.yaml"
DAFE_YAML = ROOT / "configs" / "models" / "digisteel_c3k2.yaml"
RUNS_DIR = ROOT / "runs" / "detect"
EVALS_DIR = ROOT / "evals"
EVALS_DIR.mkdir(parents=True, exist_ok=True)

# Transfer learning weights (79.4% mAP) — source for weight transfer
TL_BEST_PT = RUNS_DIR / "tl_stage2_fine_tuning" / "weights" / "best.pt"

# DAFE experiment paths (Option B: freeze stem, train DAFE + downstream)
DAFE_RUN_NAME = "tl_dafe_option_b"
DAFE_BEST_PT = RUNS_DIR / DAFE_RUN_NAME / "weights" / "best.pt"
DAFE_RESULTS_CSV = RUNS_DIR / DAFE_RUN_NAME / "results.csv"

# Verify
print(f"{'='*60}")
print("  TRANSFER LEARNING + DAFE — OPTION B")
print("  Freeze stem (0-2) | Train DAFE + downstream (3-24)")
print(f"{'='*60}")
print(f"  Device:       {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"  CUDA:         {torch.cuda.is_available()}")
print(f"  Ultralytics:  {ultralytics.__version__}")
print(f"  Data YAML:    {DATA_YAML} (exists: {DATA_YAML.exists()})")
print(f"  DAFE YAML:    {DAFE_YAML} (exists: {DAFE_YAML.exists()})")
print(f"  TL weights:   {TL_BEST_PT} (exists: {TL_BEST_PT.exists()})")
print(f"  DAFE output:  {DAFE_BEST_PT}")
print(f"{'='*60}")

In [ ]:
# ============================================================
# Cell 3: Load Transfer Learning Reference (79.4% mAP)
# ============================================================
from ultralytics import YOLO

print("=" * 70)
print("  TRANSFER LEARNING REFERENCE (79.4% mAP)")
print("=" * 70)

if not TL_BEST_PT.exists():
    print(f"  ERROR: Transfer learning weights not found: {TL_BEST_PT}")
    print("  Run transfer_learning_fresh_baseline.ipynb first.")
else:
    tl_model = YOLO(str(TL_BEST_PT))
    print(f"\n  Loaded: {TL_BEST_PT}")
    print(f"  Parameters: {sum(p.numel() for p in tl_model.model.parameters()):,}")
    print(f"  Layers:     {len(list(tl_model.model.model.children()))}")

    # Show layer structure
    print(f"\n{'Idx':<5} {'Type':<20} {'Params':>12}")
    print("-" * 40)
    for i, layer in enumerate(tl_model.model.model.children()):
        ltype = type(layer).__name__
        nparams = sum(p.numel() for p in layer.parameters())
        print(f"{i:<5} {ltype:<20} {nparams:>12,}")

    # Load reference metrics
    tl_results_path = EVALS_DIR / "tl_stage2_results.json"
    if tl_results_path.exists():
        with open(tl_results_path) as f:
            tl_ref = json.load(f)
        print(f"\n  Reference Results:")
        print(f"    mAP@0.5:      {tl_ref['mAP50']*100:.1f}%")
        print(f"    mAP@0.5:0.95: {tl_ref['mAP50_95']*100:.1f}%")
        print(f"    Precision:    {tl_ref['precision']*100:.1f}%")
        print(f"    Recall:       {tl_ref['recall']*100:.1f}%")
    else:
        print(f"\n  WARNING: TL results not found. Using 79.4% as reference.")
        tl_ref = {'mAP50': 0.794, 'mAP50_95': 0.44, 'precision': 0.733, 'recall': 0.763}

print("\n" + "=" * 70)

In [ ]:
# ============================================================
# Cell 4: Build DAFE Architecture + Transfer TL Weights (79.4%)
# ============================================================
from ultralytics import YOLO
import ultralytics.nn.tasks as tasks
from digisteel.modules import DAFE

print("=" * 70)
print("  BUILD DAFE MODEL + TRANSFER TL WEIGHTS (79.4%)")
print("  Architecture: digisteel_c3k2.yaml (C3k2 blocks match TL model)")
print("=" * 70)

# Step 1: Register DAFE
print("\n[Step 1] Registering DAFE module...")
tasks.DAFE = DAFE
print("  OK: DAFE registered")

# Step 2: Build DAFE architecture (C3k2 variant — matches TL model!)
print("\n[Step 2] Building DAFE architecture (C3k2 variant)...")
dafe_model = YOLO(str(DAFE_YAML))
print(f"  Architecture: digisteel_c3k2.yaml")
print(f"  Parameters:   {sum(p.numel() for p in dafe_model.model.parameters()):,}")
print(f"  Layers:       {len(list(dafe_model.model.model.children()))}")

# Step 3: Show architecture BEFORE loading weights
print("\n[Step 3] Architecture (before weight transfer):")
print(f"\n  {'Idx':<5} {'Type':<20} {'Params':>12}")
print("  " + "-" * 40)
for i, layer in enumerate(dafe_model.model.model.children()):
    ltype = type(layer).__name__
    nparams = sum(p.numel() for p in layer.parameters())
    print(f"  {i:<5} {ltype:<20} {nparams:>12,}")

# Step 4: Transfer TL weights (79.4%) — NOT COCO pretrained!
print("\n[Step 4] Transferring TL weights (79.4% mAP model)...")
print(f"  Source: {TL_BEST_PT}")
print("  Ultralytics auto-matches C3k2 layers (same architecture!)")

# Load TL weights into DAFE model
# Since both use C3k2, Ultralytics will match:
#   - Conv layers (stem, downsample) → MATCH
#   - C3k2 layers (backbone + neck) → MATCH
#   - SPPF → MATCH
#   - DAFE → NO MATCH (random init — new module)
#   - Detect → NO MATCH (random init — different nc)
dafe_model.load(str(TL_BEST_PT))

# Step 5: Verify transfer
print("\n[Step 5] After weight transfer:")
print(f"  Total params:     {sum(p.numel() for p in dafe_model.model.parameters()):,}")
print(f"  Trainable params: {sum(p.numel() for p in dafe_model.model.parameters() if p.requires_grad):,}")

print(f"\n  Layer status:")
print(f"  {'Idx':<5} {'Type':<20} {'Params':>12} {'Status':>18}")
print("  " + "-" * 60)
for i, layer in enumerate(dafe_model.model.model.children()):
    ltype = type(layer).__name__
    nparams = sum(p.numel() for p in layer.parameters())
    if ltype == 'DAFE':
        status = 'NEW (random init)'
    elif ltype == 'Detect':
        status = 'NEW (random init)'
    else:
        status = 'TRANSFERRED (79.4%)'
    print(f"  {i:<5} {ltype:<20} {nparams:>12,} {status:>18}")

# Count transferred vs new
transferred = sum(1 for i, l in enumerate(dafe_model.model.model.children())
                  if type(l).__name__ not in ('DAFE', 'Detect'))
new = sum(1 for i, l in enumerate(dafe_model.model.model.children())
          if type(l).__name__ in ('DAFE', 'Detect'))
print(f"\n  Summary:")
print(f"  - Transferred: {transferred} layers from TL model (79.4% mAP)")
print(f"  - New (random): {new} layers (DAFE + Detect)")
print(f"  - DAFE will learn defect-aware features on top of strong baseline!")

print("\n" + "=" * 70)

In [ ]:
# ============================================================
# Cell 5: Freeze Strategy — Option B
# ============================================================
from ultralytics import YOLO
import ultralytics.nn.tasks as tasks
from digisteel.modules import DAFE

print("=" * 70)
print("  FREEZE STRATEGY — OPTION B")
print("=" * 70)

# Rebuild model (cell-independent)
tasks.DAFE = DAFE
dafe_model = YOLO(str(DAFE_YAML))
dafe_model.load(str(TL_BEST_PT))

# Architecture:
#   Layer 0:  Conv   (stem)          ← FREEZE (universal edge detectors)
#   Layer 1:  Conv   (P2 downsample) ← FREEZE (universal)
#   Layer 2:  C3k2   (P2 features)   ← FREEZE (universal)
#   ─── DAFE boundary ───
#   Layer 3:  DAFE   (P2 enhance)    ← TRAIN (new module, needs to learn)
#   Layer 4:  Conv   (P3 downsample) ← TRAIN (needs to adapt to DAFE output)
#   Layer 5:  C3k2   (P3 features)   ← TRAIN (needs to adapt)
#   Layer 6:  DAFE   (P3 enhance)    ← TRAIN (new module)
#   Layer 7:  Conv   (P4 downsample) ← TRAIN (needs to adapt)
#   Layer 8:  C3k2   (P4 features)   ← TRAIN (needs to adapt)
#   Layer 9:  Conv   (P5 downsample) ← TRAIN
#   Layer 10: C3k2   (P5 features)   ← TRAIN
#   Layer 11: SPPF   (pooling)       ← TRAIN
#   Layer 12-23: Neck (PAN-FPN)      ← TRAIN (multi-scale fusion)
#   Layer 24: Detect (head)          ← TRAIN (classification + bbox)

FREEZE_UP_TO = 2  # Freeze layers 0-2 only (stem + early conv + C3k2)

frozen_params = 0
trainable_params = 0

for name, param in dafe_model.model.named_parameters():
    parts = name.split('.')
    if parts[0] == 'model' and len(parts) > 1:
        try:
            layer_idx = int(parts[1])
        except ValueError:
            layer_idx = -1
        if layer_idx <= FREEZE_UP_TO:
            param.requires_grad = False
            frozen_params += param.numel()
        else:
            param.requires_grad = True
            trainable_params += param.numel()
    else:
        param.requires_grad = True
        trainable_params += param.numel()

print(f"\n  Freeze boundary: layers 0-{FREEZE_UP_TO}")
print(f"  Frozen:     {frozen_params:>12,} params (stem + early features)")
print(f"  Trainable:  {trainable_params:>12,} params (DAFE + downstream + neck + head)")
print(f"  Frozen %:   {frozen_params/(frozen_params+trainable_params)*100:.1f}%")

# Verify
print(f"\n  {'Idx':<5} {'Type':<20} {'Params':>12} {'Status':>15}")
print("  " + "-" * 55)
for i, layer in enumerate(dafe_model.model.model.children()):
    ltype = type(layer).__name__
    nparams = sum(p.numel() for p in layer.parameters())
    is_frozen = all(not p.requires_grad for p in layer.parameters()) if list(layer.parameters()) else True
    status = 'LOCK FROZEN' if is_frozen else 'FIRE TRAINABLE'
    tag = ''
    if ltype == 'DAFE':
        tag = ' (NEW)'
    elif ltype == 'Detect':
        tag = ' (NEW)'
    elif i <= FREEZE_UP_TO:
        tag = ' (universal)'
    print(f"  {i:<5} {ltype:<20} {nparams:>12,} {status:>15}{tag}")

print(f"\n  Why this works:")
print(f"  - Layers 0-2 FROZEN: Universal edge/texture detectors (COCO pretrained)")
print(f"  - DAFE TRAINS: Learns defect-aware enhancement")
print(f"  - Downstream (4-11) TRAINS: Adapts to DAFE's new feature maps")
print(f"  - Neck TRAINS: Learns defect-specific multi-scale fusion")
print(f"  - Head TRAINS: Learns 6-class mapping")
print(f"  - DAFE and backbone CO-ADAPT (like exp_5a, but with better init)")

print("\n" + "=" * 70)

In [ ]:
# ============================================================
# Cell 6: Train DAFE — Option B (Freeze stem, train DAFE + downstream)
# ============================================================
from ultralytics import YOLO
import ultralytics.nn.tasks as tasks
from digisteel.modules import DAFE
import time

print("=" * 70)
print("  TRAINING: TL + DAFE — OPTION B")
print("  Freeze: 0-2 (stem) | Train: 3-24 (DAFE + downstream + neck + head)")
print("=" * 70)

# Rebuild model (cell-independent)
tasks.DAFE = DAFE
dafe_model = YOLO(str(DAFE_YAML))
dafe_model.load(str(TL_BEST_PT))
print(f"  Loaded TL weights from: {TL_BEST_PT}")

# Apply freezing: freeze stem only (0-2), train everything else (3-24)
FREEZE_UP_TO = 2
for name, param in dafe_model.model.named_parameters():
    parts = name.split('.')
    if parts[0] == 'model' and len(parts) > 1:
        try:
            layer_idx = int(parts[1])
        except ValueError:
            layer_idx = -1
        if layer_idx <= FREEZE_UP_TO:
            param.requires_grad = False
        else:
            param.requires_grad = True
    else:
        param.requires_grad = True

frozen = sum(p.numel() for p in dafe_model.model.parameters() if not p.requires_grad)
trainable = sum(p.numel() for p in dafe_model.model.parameters() if p.requires_grad)
print(f"  Frozen: {frozen:,} params (layers 0-{FREEZE_UP_TO})")
print(f"  Trainable: {trainable:,} params (layers {FREEZE_UP_TO+1}-24)")

# Training recipe (SAME as exp_5a)
dafe_overrides = {
    'data': str(DATA_YAML),
    'task': 'detect',
    'epochs': 600,
    'patience': 150,
    'batch': 16,
    'imgsz': 800,
    'device': 0,
    'optimizer': 'AdamW',
    'lr0': 0.001,
    'lrf': 0.01,
    'momentum': 0.937,
    'weight_decay': 0.0005,
    'warmup_epochs': 5,
    'warmup_momentum': 0.8,
    'warmup_bias_lr': 0.1,
    'mosaic': 0.0,
    'mixup': 0.15,
    'copy_paste': 0.1,
    'copy_paste_mode': 'flip',
    'degrees': 10.0,
    'translate': 0.2,
    'scale': 0.6,
    'shear': 5.0,
    'perspective': 0.0,
    'flipud': 0.5,
    'fliplr': 0.5,
    'hsv_h': 0.015,
    'hsv_s': 0.7,
    'hsv_v': 0.4,
    'erasing': 0.4,
    'cos_lr': True,
    'deterministic': True,
    'close_mosaic': 10,
    'amp': True,
    'seed': 42,
    'workers': 8,
    'save': True,
    'save_period': 50,
    'plots': True,
    'project': str(RUNS_DIR),
    'name': DAFE_RUN_NAME,
    'exist_ok': True,
    'verbose': True,
}

print(f"\n  Recipe:")
print(f"  Epochs:    {dafe_overrides['epochs']}")
print(f"  LR:        {dafe_overrides['lr0']} (cosine decay)")
print(f"  Patience:  {dafe_overrides['patience']}")
print(f"  imgsz:     {dafe_overrides['imgsz']}")

print(f"\n  Strategy: DAFE + backbone downstream CO-ADAPT")
print(f"  - Stem frozen (universal edge detectors)")
print(f"  - DAFE learns defect-aware enhancement")
print(f"  - Downstream layers adapt to DAFE's output")
print(f"  - Like exp_5a, but stem is protected from disruption")

# Train
dafe_start = time.time()
dafe_results = dafe_model.train(**dafe_overrides)
dafe_time = time.time() - dafe_start

print(f"\n{'='*60}")
print(f"  TRAINING COMPLETE")
print(f"  Duration: {dafe_time/3600:.1f} hours")
print(f"  Weights:  {DAFE_BEST_PT}")
print(f"{'='*60}")

In [ ]:
# ============================================================
# Cell 7: Evaluate DAFE on TEST Set
# ============================================================
from ultralytics import YOLO
import ultralytics.nn.tasks as tasks
from digisteel.modules import DAFE
import json

print("=" * 70)
print("  TL + DAFE: TEST SET EVALUATION")
print("=" * 70)

if not DAFE_BEST_PT.exists():
    print(f"  ERROR: Weights not found: {DAFE_BEST_PT}")
    print("  Run Cell 6 first.")
else:
    tasks.DAFE = DAFE
    dafe_best = YOLO(str(DAFE_BEST_PT))

    print(f"\n  Evaluating on TEST set...")
    dafe_metrics = dafe_best.val(
        data=str(DATA_YAML),
        split='test',
        imgsz=800,
        batch=16,
        device=0,
        plots=True,
        verbose=True,
    )

    dafe_map50 = float(dafe_metrics.box.map50)
    dafe_map5095 = float(dafe_metrics.box.map)
    dafe_prec = float(dafe_metrics.box.mp)
    dafe_rec = float(dafe_metrics.box.mr)

    class_names = dafe_best.names
    dafe_per_class = {}
    for cid, cname in class_names.items():
        if cid < len(dafe_metrics.box.ap50):
            dafe_per_class[cname] = float(dafe_metrics.box.ap50[cid])

    print(f"\n{'='*60}")
    print("TL + DAFE TEST RESULTS")
    print(f"{'='*60}")
    print(f"  mAP@0.5:      {dafe_map50:.4f} ({dafe_map50*100:.1f}%)")
    print(f"  mAP@0.5:0.95: {dafe_map5095:.4f} ({dafe_map5095*100:.1f}%)")
    print(f"  Precision:    {dafe_prec:.4f} ({dafe_prec*100:.1f}%)")
    print(f"  Recall:       {dafe_rec:.4f} ({dafe_rec*100:.1f}%)")
    print(f"\n  Per-class AP@0.5:")
    for cname in sorted(dafe_per_class):
        bar = '█' * int(dafe_per_class[cname] * 40)
        print(f"    {cname:<18} {dafe_per_class[cname]*100:.1f}% {bar}")

    dafe_eval = {
        "experiment": "tl_plus_dafe_v2",
        "mAP50": dafe_map50,
        "mAP50_95": dafe_map5095,
        "precision": dafe_prec,
        "recall": dafe_rec,
        "per_class": dafe_per_class,
    }
    with open(EVALS_DIR / "tl_dafe_v2_results.json", "w") as f:
        json.dump(dafe_eval, f, indent=2)
    print(f"\n  Saved: {EVALS_DIR / 'tl_dafe_v2_results.json'}")

print("\n" + "=" * 70)

In [ ]:
# ============================================================
# Cell 8: Loss Plots + Overfitting Analysis
# ============================================================
import matplotlib.pyplot as plt
import pandas as pd

print("=" * 70)
print("  TL + DAFE: LOSS ANALYSIS & OVERFITTING DETECTION")
print("=" * 70)

if not DAFE_RESULTS_CSV.exists():
    print(f"  ERROR: results.csv not found: {DAFE_RESULTS_CSV}")
else:
    df = pd.read_csv(DAFE_RESULTS_CSV)
    df.columns = df.columns.str.strip()
    print(f"  Loaded: {len(df)} epochs")

    all_loss_cols = [c for c in df.columns if 'loss' in c.lower()]
    train_total = [c for c in df.columns if 'train/box_loss' in c.lower() or c.lower() == 'train/loss']
    val_total = [c for c in df.columns if 'val/box_loss' in c.lower() or c.lower() == 'val/loss']

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('TL + DAFE v2: Training Analysis', fontsize=14, fontweight='bold')

    # Plot 1: All loss curves
    ax = axes[0, 0]
    for col in all_loss_cols:
        ax.plot(df[col], label=col, alpha=0.8)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('All Loss Curves')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

    # Plot 2: Train vs Val loss
    ax = axes[0, 1]
    if train_total:
        ax.plot(df[train_total[0]], label='Train Loss', color='blue', linewidth=2)
    if val_total:
        ax.plot(df[val_total[0]], label='Val Loss', color='red', linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Train vs Val Loss (Overfitting Check)')
    ax.legend()
    ax.grid(True, alpha=0.3)

    if train_total and val_total:
        gap = df[val_total[0]].values - df[train_total[0]].values
        final_gap = gap[-1]
        color = 'red' if final_gap > 0.5 else 'green'
        label = 'OVERFITTING' if final_gap > 0.5 else 'OK'
        ax.annotate(f'{label}\nGap={final_gap:.2f}',
                   xy=(len(gap)-1, df[val_total[0]].values[-1]),
                   fontsize=10, color=color, fontweight='bold')

    # Plot 3: mAP curves
    ax = axes[1, 0]
    map50_col = [c for c in df.columns if 'map50' in c.lower() and 'map50-' not in c.lower()]
    map_col = [c for c in df.columns if 'map50-95' in c.lower()]
    if map50_col:
        ax.plot(df[map50_col[0]], label='mAP@0.5', color='blue', linewidth=2)
    if map_col:
        ax.plot(df[map_col[0]], label='mAP@0.5:0.95', color='red', linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('mAP')
    ax.set_title('mAP Progress')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Plot 4: Overfitting Gap
    ax = axes[1, 1]
    if train_total and val_total:
        gap = df[val_total[0]].values - df[train_total[0]].values
        ax.plot(gap, color='purple', linewidth=2)
        ax.axhline(y=0, color='green', linestyle='--', alpha=0.5, label='Perfect fit')
        ax.fill_between(range(len(gap)), 0, gap, alpha=0.3,
                       color='red' if gap[-1] > 0 else 'green')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Val - Train Loss')
        ax.set_title('Overfitting Gap (Val - Train)')
        ax.legend()
        ax.grid(True, alpha=0.3)

        if gap[-1] > 1.0:
            diagnosis = "OVERFITTING"
        elif gap[-1] > 0.5:
            diagnosis = "MILD OVERFITTING"
        elif gap[-1] < -0.5:
            diagnosis = "UNDERFITTING"
        else:
            diagnosis = "GOOD FIT"
        ax.set_xlabel(f'Epoch — {diagnosis}')

    plt.tight_layout()
    loss_path = EVALS_DIR / 'tl_dafe_v2_loss_analysis.png'
    plt.savefig(loss_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"\n  Saved: {loss_path}")

print("\n" + "=" * 70)

In [ ]:
# ============================================================
# Cell 9: Comparison — TL Only vs TL + DAFE vs exp_5a
# ============================================================
import json
import matplotlib.pyplot as plt
import numpy as np

print("=" * 70)
print("  FINAL COMPARISON")
print("=" * 70)

# Load all results
tl_path = EVALS_DIR / "tl_stage2_results.json"
dafe_v2_path = EVALS_DIR / "tl_dafe_v2_results.json"
exp5a_path = EVALS_DIR / "exp_5a_digisteel_v4_dafe.json"

tl = json.load(open(tl_path)) if tl_path.exists() else None
dafe_v2 = json.load(open(dafe_v2_path)) if dafe_v2_path.exists() else None
exp5a = json.load(open(exp5a_path)) if exp5a_path.exists() else None

# Baseline
BASELINE = {"mAP50": 0.788, "mAP50_95": 0.452, "precision": 0.719, "recall": 0.763}

if tl and dafe_v2:
    print(f"\n  {'Metric':<20} {'Baseline':>10} {'TL Only':>10} {'TL+DAFE':>10} {'exp_5a':>10}")
    print("  " + "-" * 65)

    exp5a_vals = {
        'mAP50': exp5a['map50'] if exp5a else 0,
        'mAP50_95': exp5a['map50_95'] if exp5a else 0,
        'precision': exp5a['precision'] if exp5a else 0,
        'recall': exp5a['recall'] if exp5a else 0,
    }

    for label, key in [('mAP@0.5', 'mAP50'), ('mAP@0.5:0.95', 'mAP50_95'),
                       ('Precision', 'precision'), ('Recall', 'recall')]:
        b = BASELINE[key] * 100
        v1 = tl[key] * 100
        v2 = dafe_v2[key] * 100
        v3 = exp5a_vals[key] * 100
        print(f"  {label:<20} {b:>9.1f}% {v1:>9.1f}% {v2:>9.1f}% {v3:>9.1f}%")

    # DAFE impact
    print(f"\n  DAFE Impact (TL+DAFE vs TL Only):")
    print(f"  {'Metric':<20} {'TL Only':>10} {'TL+DAFE':>10} {'Delta':>10}")
    print("  " + "-" * 55)
    for label, key in [('mAP@0.5', 'mAP50'), ('mAP@0.5:0.95', 'mAP50_95'),
                       ('Precision', 'precision'), ('Recall', 'recall')]:
        v1 = tl[key] * 100
        v2 = dafe_v2[key] * 100
        d = v2 - v1
        sign = '+' if d > 0 else ''
        print(f"  {label:<20} {v1:>9.1f}% {v2:>9.1f}% {sign}{d:>9.1f}%")

    # Per-class
    if exp5a and 'per_class_ap50' in exp5a:
        print(f"\n  Per-class AP@0.5:")
        print(f"  {'Class':<20} {'TL Only':>10} {'TL+DAFE':>10} {'exp_5a':>10}")
        print("  " + "-" * 55)
        for cname in sorted(tl['per_class'].keys()):
            v1 = tl['per_class'][cname] * 100
            v2 = dafe_v2['per_class'][cname] * 100
            v3 = exp5a['per_class_ap50'].get(cname, 0) * 100
            print(f"  {cname:<20} {v1:>9.1f}% {v2:>9.1f}% {v3:>9.1f}%")

    # Visual
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    labels = ['mAP@0.5', 'mAP@0.5:0.95', 'Precision', 'Recall']
    baseline_vals = [BASELINE[k]*100 for k in ['mAP50','mAP50_95','precision','recall']]
    tl_vals = [tl[k]*100 for k in ['mAP50','mAP50_95','precision','recall']]
    dafe_vals = [dafe_v2[k]*100 for k in ['mAP50','mAP50_95','precision','recall']]
    exp5a_vals_plot = [exp5a_vals[k]*100 for k in ['mAP50','mAP50_95','precision','recall']]

    x = np.arange(len(labels))
    w = 0.2

    axes[0].bar(x - 1.5*w, baseline_vals, w, label='Baseline', color='#95a5a6')
    axes[0].bar(x - 0.5*w, tl_vals, w, label='TL Only', color='#3498db')
    axes[0].bar(x + 0.5*w, dafe_vals, w, label='TL+DAFE v2', color='#e74c3c')
    axes[0].bar(x + 1.5*w, exp5a_vals_plot, w, label='exp_5a DAFE', color='#2ecc71')
    axes[0].set_ylabel('Score (%)')
    axes[0].set_title('Overall Metrics Comparison', fontweight='bold')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(labels)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3, axis='y')

    # Per-class
    classes = sorted(tl['per_class'].keys())
    tl_c = [tl['per_class'][c]*100 for c in classes]
    dafe_c = [dafe_v2['per_class'][c]*100 for c in classes]
    exp5a_c = [exp5a['per_class_ap50'].get(c, 0)*100 for c in classes] if exp5a else [0]*len(classes)

    x2 = np.arange(len(classes))
    axes[1].bar(x2 - w, tl_c, w, label='TL Only', color='#3498db')
    axes[1].bar(x2, dafe_c, w, label='TL+DAFE v2', color='#e74c3c')
    axes[1].bar(x2 + w, exp5a_c, w, label='exp_5a DAFE', color='#2ecc71')
    axes[1].set_ylabel('AP@0.5 (%)')
    axes[1].set_title('Per-Class Comparison', fontweight='bold')
    axes[1].set_xticks(x2)
    axes[1].set_xticklabels(classes, rotation=45, ha='right')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    comp_path = EVALS_DIR / 'tl_dafe_v2_comparison.png'
    plt.savefig(comp_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"\n  Saved: {comp_path}")

    # Save
    final = {
        "baseline": BASELINE,
        "tl_only": tl,
        "tl_plus_dafe_v2": dafe_v2,
        "exp_5a": exp5a,
        "dafe_impact": {
            "mAP50_delta": dafe_v2['mAP50'] - tl['mAP50'],
            "mAP50_95_delta": dafe_v2['mAP50_95'] - tl['mAP50_95'],
        },
    }
    with open(EVALS_DIR / "tl_dafe_v2_final_comparison.json", "w") as f:
        json.dump(final, f, indent=2)
    print(f"  Saved: {EVALS_DIR / 'tl_dafe_v2_final_comparison.json'}")

else:
    print("\n  Results not found. Run Cell 7 first.")

print("\n" + "=" * 70)

---

## Summary

### What This Notebook Does

| Step | Description |
|------|-------------|
| 1 | Load TL reference model (79.4% mAP) |
| 2 | Build DAFE architecture (C3k2 variant — matches TL model!) |
| 3 | Transfer TL weights into DAFE model (C3k2 layers match!) |
| 4 | Freeze early backbone (0-6), unfreeze DAFE + neck + head (7-24) |
| 5 | Train with optimized recipe (same as exp_5a: 600 epochs, lr=0.001) |
| 6 | Evaluate on TEST set |
| 7 | Compare: Baseline vs TL vs TL+DAFE vs exp_5a |

### Why This Should Beat exp_5a

| | exp_5a | TL + DAFE (this notebook) |
|---|---|---|
| **Backbone init** | COCO pretrained (yolo11n.pt) | TL model (79.4% — already fine-tuned on NEU-DET!) |
| **DAFE init** | Random | Random |
| **Architecture** | C2f + DAFE | C3k2 + DAFE (matches TL model) |
| **Weight transfer** | COCO → C2f (partial match) | TL → C3k2 (full match!) |
| **Expected mAP** | 80.3% | **80-83%** (better backbone!) |

### Key Insight

The TL model (79.4%) has backbone weights that are **already adapted to steel defects**. When we transfer these into the DAFE architecture:

- Conv layers → MATCH (edge/texture detectors already tuned for steel)
- C3k2 layers → MATCH (feature extractors already learned steel patterns)
- SPPF → MATCH (pooling already adapted)
- DAFE → NEW (random init — will learn defect-aware enhancement)
- Detect → NEW (random init — will learn 6-class mapping)

DAFE starts from a **much better initialization** than exp_5a's COCO pretrained weights. The backbone already "knows" steel defects — DAFE just needs to learn how to enhance them.

### Files Generated

```
configs/models/digisteel_c3k2.yaml  ← NEW: C3k2 + DAFE architecture

runs/detect/tl_dafe_c3k2_transfer/weights/best.pt

evals/
├── tl_dafe_v2_results.json
├── tl_dafe_v2_loss_analysis.png
├── tl_dafe_v2_comparison.png
└── tl_dafe_v2_final_comparison.json
```